In [1]:
"""
gap11_final_memsafe.py
======================
Memory-safe version. Three measures for A, one for E.
Runs in <10 seconds on standard Colab.
"""

import numpy as np

N = 9
FANO = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]


def build_G(lam=1.0, t_sq=-1.0):
    M = np.zeros((N, N, N), dtype=float)
    for i in range(N):
        M[7, i, i] = 1.0
        M[i, 7, i] = 1.0
    for (a, b, c) in FANO:
        M[a, b, c] = 1.0; M[b, c, a] = 1.0; M[c, a, b] = 1.0
        M[b, a, c] = -1.0; M[c, b, a] = -1.0; M[a, c, b] = -1.0
    for i in range(7):
        M[i, i, 7] = -1.0
    M[8, 8, 7] = t_sq
    for i in range(7):
        M[8, i, i] = lam
        M[i, 8, i] = -lam
    M[8, 7, 7] = lam
    M[7, 8, 8] = 1.0
    return M


def mul(M, x, y):
    return np.einsum('i,j,ijk->k', x, y, M, optimize=True)


def unit(i):
    v = np.zeros(N); v[i] = 1.0
    return v


def option_A(M, name):
    print("=" * 72)
    print(f"OPTION A: {name}")
    print("=" * 72)
    print()

    t_vec = unit(8)
    x = unit(7) - t_vec
    print(f"Canonical G-dark element: x = 1 - t")
    print()

    # M1: support
    support = int(np.sum(np.abs(x) > 1e-10))
    sub1 = (10 - support) / 10
    print(f"M1 (support):         {support} slots, sub-channel = {sub1:.4f}")

    # M2: t-orbit (capped)
    orbit = []
    cur = x.copy()
    for _ in range(50):
        if any(np.allclose(cur, o, atol=1e-6) for o in orbit):
            break
        orbit.append(cur.copy())
        cur = mul(M, t_vec, cur)
    sub2 = (10 - len(orbit)) / 10
    print(f"M2 (t-orbit):         {len(orbit)} elements, sub-channel = {sub2:.4f}")

    # M3: Fano annihilation
    killed = sum(1 for i in range(7) if np.allclose(mul(M, x, unit(i)), 0, atol=1e-8))
    sub3 = (10 - 9) / 10  # 7 Fano + 2 (identity+t) = 9 consumed
    print(f"M3 (Fano annihilated):{killed}/7 left-killed, sub-channel = {sub3:.4f}")

    print()
    print(f"Spread: {sub1:.2f} to {sub3:.2f} (8× range)")
    print(f"Electron reference: 0.200")
    print()


def option_E(M, name, n_samples=2000):
    print("=" * 72)
    print(f"OPTION E: {name}")
    print("=" * 72)
    print()
    rng = np.random.default_rng(42)
    fano_units = [unit(i) for i in range(7)]

    n_left = n_right = n_both = 0
    for _ in range(n_samples):
        x = rng.normal(size=N)
        nrm = np.linalg.norm(x)
        if nrm < 1e-10: continue
        x /= nrm
        L = all(np.linalg.norm(mul(M, x, e)) < 1e-6 for e in fano_units)
        R = all(np.linalg.norm(mul(M, e, x)) < 1e-6 for e in fano_units)
        if L: n_left += 1
        if R: n_right += 1
        if L and R: n_both += 1

    print(f"Left-dark:  {n_left}/{n_samples} = {100*n_left/n_samples:.4f}%")
    print(f"Right-dark: {n_right}/{n_samples} = {100*n_right/n_samples:.4f}%")
    print(f"Both-dark:  {n_both}/{n_samples} = {100*n_both/n_samples:.4f}%")
    print()
    print(f"Analytic dimension estimate:")
    print(f"  In ℝ⁹, dark sector = 2D (identity + t)")
    print(f"  Visible (Fano) = 7D")
    print(f"  Ratio 2/7 = {2/7:.4f}")
    print(f"  Observed Ω_DM/Ω_b ≈ 5.4")
    print(f"  Discrepancy: {5.4/(2/7):.1f}×")
    print()


M_G = build_G(lam=1.0)
option_A(M_G, "G (λ=1)")
option_E(M_G, "G (λ=1)")

OPTION A: G (λ=1)

Canonical G-dark element: x = 1 - t

M1 (support):         2 slots, sub-channel = 0.8000
M2 (t-orbit):         6 elements, sub-channel = 0.4000
M3 (Fano annihilated):7/7 left-killed, sub-channel = 0.1000

Spread: 0.80 to 0.10 (8× range)
Electron reference: 0.200

OPTION E: G (λ=1)

Left-dark:  0/2000 = 0.0000%
Right-dark: 0/2000 = 0.0000%
Both-dark:  0/2000 = 0.0000%

Analytic dimension estimate:
  In ℝ⁹, dark sector = 2D (identity + t)
  Visible (Fano) = 7D
  Ratio 2/7 = 0.2857
  Observed Ω_DM/Ω_b ≈ 5.4
  Discrepancy: 18.9×

